# GRPO fine-tuning (Unsloth) — AEGIS-Env against your Hugging Face Space

This notebook trains a **small language model** with **GRPO** using rewards from **[AEGIS-Env](https://huggingface.co/spaces/NishithP2004/aegis-env)**.

- **Remote environment:** OpenEnv WebSocket API at `…/openenv` on the Space.
- **Three difficulty tiers:** `easy`, `medium`, `hard` → same mapping as the server (`mohler` / `asap-sas` / `ricechem` datasets).
- **Reward signal:** After a **fixed deterministic replay** through arbiter → scrutinizer → validator, the model’s completion is parsed as a **mentor-stage** JSON action (`AegisAction`); the Space returns the **final step reward** in `[0, 1]`.

**Requirements:** GPU runtime (e.g. T4). The Space must be awake (cold start can take a minute). The clone step pulls `client.py`, `models.py`, and `inference.py` from your Space repo for `build_user_prompt` / `_parse_action`.


In [ ]:
# %%capture
import os, importlib.util
!pip install -qqq uv nest_asyncio
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except Exception:
        get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install -qqq "openenv-core[core]>=0.2.2" openai pydantic


In [ ]:
import os
# AEGIS-Env code (client + prompts). Override with your Git repo URL if preferred.
AEGIS_REPO = os.environ.get(
    "AEGIS_CODE_CLONE_URL",
    "https://huggingface.co/spaces/NishithP2004/aegis-env",
)
!rm -rf /content/aegis-env-space && git clone --depth 1 {AEGIS_REPO} /content/aegis-env-space

import sys
sys.path.insert(0, "/content/aegis-env-space")

import nest_asyncio
nest_asyncio.apply()

AEGIS_OPENENV_BASE = os.environ.get(
    "AEGIS_OPENENV_BASE",
    "https://huggingface.co/spaces/NishithP2004/aegis-env/openenv",
)
print("OpenEnv WS/HTTP base:", AEGIS_OPENENV_BASE)


In [ ]:
import asyncio
from typing import Any, List

from datasets import Dataset
from transformers import TextStreamer
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

from models import AegisAction, AegisObservation
from client import AegisEnv
from inference import build_user_prompt, _parse_action, _strip_markdown_json_fence


async def replay_to_mentor_obs(env: AegisEnv, seed: int, task_name: str) -> AegisObservation:
    r = await env.reset(seed=seed, task_name=task_name)
    obs = r.observation
    ms = float(obs.max_score or 1.0)
    mid = max(0.0, min(ms, ms * 0.5))
    a1 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Arbiter: preliminary assessment aligned with the rubric.",
        routing_decision="proceed",
    )
    obs = (await env.step(a1)).observation
    a2 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Scrutinizer: refined criterion-by-criterion review.",
        routing_decision="proceed",
    )
    obs = (await env.step(a2)).observation
    a3 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Validator: consistency check passed; proceed to mentor.",
        routing_decision="proceed",
    )
    obs = (await env.step(a3)).observation
    if str(getattr(obs, "current_stage", "") or "") != "mentor":
        raise RuntimeError(f"Expected mentor stage, got {getattr(obs, 'current_stage', None)}")
    return obs


def _run(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


async def build_one_row(task_name: str, seed: int, base_url: str) -> dict[str, Any]:
    env = AegisEnv(base_url=base_url)
    await env.connect()
    try:
        obs = await replay_to_mentor_obs(env, seed, task_name)
        user_text = build_user_prompt(4, None, 0.0, [], obs)
        return {
            "prompt": [{"role": "user", "content": user_text}],
            "answer": 0,
            "task_name": task_name,
            "episode_seed": int(seed),
            "mentor_max_score": float(obs.max_score or 1.0),
        }
    finally:
        await env.close()


def build_dataset(
    base_url: str,
    n_per_tier: int = 40,
) -> Dataset:
    rows: List[dict] = []
    tier_seeds = {"easy": 1000, "medium": 11000, "hard": 21000}
    for task_name in ("easy", "medium", "hard"):
        base_seed = tier_seeds[task_name]
        for i in range(n_per_tier):
            seed = base_seed + i
            row = _run(build_one_row(task_name, seed, base_url))
            rows.append(row)
            print("built", task_name, i, "max_score", row["mentor_max_score"])
    return Dataset.from_list(rows)


In [ ]:
# Build prompts against the live Space (sequential; Space allows limited concurrency)
train_ds = build_dataset(AEGIS_OPENENV_BASE, n_per_tier=40)
sample = train_ds[0]["prompt"]
print(train_ds)


In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
max_seq_length = 4096
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=max_seq_length,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

maximum_length = len(tokenizer.apply_chat_template(sample, add_generation_prompt=True))
print("Prompt tokens (sample):", maximum_length)


In [ ]:
async def mentor_env_reward_one(
    completion_text: str,
    seed: int,
    task_name: str,
    mentor_max_score: float,
    base_url: str,
) -> float:
    env = AegisEnv(base_url=base_url)
    await env.connect()
    try:
        obs = await replay_to_mentor_obs(env, seed, task_name)
        ms = float(obs.max_score or mentor_max_score or 1.0)
        action = _parse_action(_strip_markdown_json_fence(completion_text), ms)
        sr = await env.step(action)
        r = float(sr.reward if sr.reward is not None else 0.0)
        return r * 20.0
    except Exception as e:
        print("mentor_env_reward error:", e)
        return -8.0
    finally:
        await env.close()


def format_valid(prompts, completions, mentor_max_score, **kwargs):
    scores = []
    for c, ms in zip(completions, mentor_max_score):
        try:
            _parse_action(_strip_markdown_json_fence(c[0]["content"]), float(ms))
            scores.append(1.0)
        except Exception:
            scores.append(-1.5)
    return scores


def no_markdown_fence(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        t = c[0]["content"].strip()
        scores.append(-0.5 if t.startswith("```") else 0.2)
    return scores


def mentor_success(prompts, completions, episode_seed, task_name, mentor_max_score, **kwargs):
    scores = []
    for c, seed, task, ms in zip(completions, episode_seed, task_name, mentor_max_score):
        text = c[0]["content"]
        coro = mentor_env_reward_one(text, int(seed), str(task), float(ms), AEGIS_OPENENV_BASE)
        scores.append(float(_run(coro)))
    return scores


In [ ]:
text = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    temperature=0.7,
    max_new_tokens=512,
    streamer=TextStreamer(tokenizer, skip_prompt=False),
)


In [ ]:
max_prompt_length = min(maximum_length + 8, max_seq_length // 2)
max_completion_length = max_seq_length - max_prompt_length

training_args = GRPOConfig(
    temperature=0.8,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=200,
    save_steps=50,
    report_to="trackio",
    output_dir="outputs_grpo_aegis_hf",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_valid, no_markdown_fence, mentor_success],
    args=training_args,
    train_dataset=train_ds,
)

trainer.train()


## Notes

- **Latency:** `mentor_success` calls your Space once per completion; keep `n_per_tier`, `max_steps`, and `num_generations` modest while iterating.
- **WebSocket client:** Use an `AegisEnv` / `client.py` that maps `current_stage`, `pipeline_history`, and `refinement_loops_taken` from observations (updated in this repo’s `client.py` — redeploy the Space to pick it up).
- **Overrides:** Set `AEGIS_OPENENV_BASE` or `AEGIS_CODE_CLONE_URL` instead of editing cells.
